# Task 1 (Easy): Netflix Data Cleaning & Preparation
**Auspify Technologies | Data Analysis Using Python Internship**
**Intern:** Aarti Jamage

**Goal:** Prepare and organize the Netflix dataset for business analytics and reporting.

**Workflow:** 1) Import → 2) Missing values → 3) Duplicates & formatting → 4) Standardize columns → 5) Export

## Step 1: Import the dataset using Python and Pandas

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Dataset.csv")
raw_rows = len(df)
print("Shape:", df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Observation:** The dataset has 8,790 rows and 10 columns. Pandas reports no nulls, but missing values appear to be stored as the text `"Not Given"`, so they must be checked manually.

## Step 2: Identify and handle missing values

In [5]:
# Pandas sees no NaN because missing data is stored as text
print("Real nulls:", df.isna().sum().sum())
placeholder = df.isin(["Not Given"]).sum()
print("\n'Not Given' placeholders per column:")
print(placeholder[placeholder > 0])

Real nulls: 0

'Not Given' placeholders per column:
director    2588
country      287
dtype: int64


In [6]:
# Convert the placeholder text into real missing values
df = df.replace("Not Given", np.nan)
print(df.isna().sum()[lambda s: s > 0])

director    2588
country      287
dtype: int64


In [7]:
# director: 2,588 missing (~29%), too many to drop, so fill with 'Unknown'
# country : 287 missing, fill with 'Unknown' to keep the rows
df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
print("Remaining nulls:", df.isna().sum().sum())

Remaining nulls: 0


**Decision:** Rows were **filled, not dropped**, because dropping them would remove about 29% of the data and bias later analysis (for example the country analysis in Task 3). "Unknown" keeps the rows while clearly labelling the gap.

## Step 3: Remove duplicate records and formatting inconsistencies

In [ ]:
print("Fully identical rows:", df.duplicated().sum())

# Same title + type + year but different show_id?
dups = df[df.duplicated(["title", "type", "release_year"], keep=False)]
dups.sort_values("title")

Fully identical rows: 0


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_value,duration_unit,rating_group


These 3 pairs are identical in every column except `show_id`, so they are true duplicate records. We ignore `show_id` and drop them.

In [15]:
before = len(df)
df = df.drop_duplicates(subset=df.columns.drop("show_id"), keep="first").reset_index(drop=True)
print(f"Rows before: {before} | after: {len(df)} | removed: {before - len(df)}")

Rows before: 8787 | after: 8786 | removed: 1


In [10]:
# Trim stray whitespace in all text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].str.strip()

# date_added: text -> real datetime, plus helper columns
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()

# duration: "90 min" / "2 Seasons" -> number + unit
dur = df["duration"].str.extract(r"(\d+)\s*(\D+)")
df["duration_value"] = dur[0].astype(int)
df["duration_unit"] = dur[1].str.strip().str.replace("Seasons", "Season")

df[["date_added", "year_added", "month_added", "duration", "duration_value", "duration_unit"]].head()

,date_added,year_added,month_added,duration,duration_value,duration_unit
0,2021-09-25,2021,September,90 min,90,min
1,2021-09-24,2021,September,1 Season,1,Season
2,2021-09-24,2021,September,1 Season,1,Season
3,2021-09-22,2021,September,91 min,91,min
4,2021-09-24,2021,September,125 min,125,min


## Step 4: Standardize columns (Country, Rating, Type)

In [11]:
df["type"] = df["type"].str.strip().str.title().replace({"Tv Show": "TV Show"})
df["country"] = df["country"].str.strip().str.title()
df["rating"] = df["rating"].str.strip().str.upper()

print(df["type"].value_counts(), "\n")
print(df["rating"].value_counts(), "\n")
print("Unique countries:", df["country"].nunique())
df["country"].value_counts().head(10)

type
Movie      6124
TV Show    2663
Name: count, dtype: int64 

rating
TV-MA       3205
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64 

Unique countries: 86


country
United States     3240
India             1056
United Kingdom     638
Pakistan           420
Unknown            287
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Name: count, dtype: int64

In [12]:
# Audience group (used later in Task 5)
rating_map = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Teens", "PG": "Teens", "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
}
df["rating_group"] = df["rating"].map(rating_map).fillna("Unrated")
df["rating_group"].value_counts()

rating_group
Adults     4006
Teens      3793
Kids        906
Unrated      82
Name: count, dtype: int64

**Note:** `NR` and `UR` ratings (not rated / unrated) are grouped as **Unrated** because they have no audience classification.

## Step 5: Export the cleaned dataset

In [13]:
print("Final shape:", df.shape)
print("Total nulls:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.info()

Final shape: (8787, 15)
Total nulls: 0
Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 8787 entries, 0 to 8786
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   show_id         8787 non-null   str           
 1   type            8787 non-null   str           
 2   title           8787 non-null   str           
 3   director        8787 non-null   str           
 4   country         8787 non-null   str           
 5   date_added      8787 non-null   datetime64[us]
 6   release_year    8787 non-null   int64         
 7   rating          8787 non-null   str           
 8   duration        8787 non-null   str           
 9   listed_in       8787 non-null   str           
 10  year_added      8787 non-null   int32         
 11  month_added     8787 non-null   str           
 12  duration_value  8787 non-null   int64         
 13  duration_unit   8787 non-null   str           
 14  rating_gro

In [14]:
df.to_csv("../data/netflix_cleaned.csv", index=False)
print("Saved: ../data/netflix_cleaned.csv") 

Saved: ../data/netflix_cleaned.csv


## Summary of cleaning

| Item | Before | After |
|---|---|---|
| Rows | 8,790 | 8,787 |
| Hidden missing values ("Not Given") | director 2,588, country 287 | filled with "Unknown" |
| Duplicate records | 3 | 0 |
| `date_added` | text | datetime (+ `year_added`, `month_added`) |
| `duration` | text | `duration_value` + `duration_unit` |
| New column | none | `rating_group` (Kids / Teens / Adults / Unrated) |

The cleaned file `netflix_cleaned.csv` is ready for Tasks 2 to 6.